### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="taiwanese_bankruptcy_prediction",
    dataset_year="2009",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5004D",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/taiwanese_bankruptcy_prediction/ && wget -P local-data-warehouse/taiwanese_bankruptcy_prediction/ https://archive.ics.uci.edu/static/public/572/taiwanese+bankruptcy+prediction.zip && unzip local-data-warehouse/taiwanese_bankruptcy_prediction/taiwanese+bankruptcy+prediction.zip -d local-data-warehouse/taiwanese_bankruptcy_prediction/ && rm local-data-warehouse/taiwanese_bankruptcy_prediction/taiwanese+bankruptcy+prediction.zip
""",
    # References
    academic_reference_bibtex="""@article{liang2016financial,
  title={Financial ratios and corporate governance indicators in bankruptcy prediction: A comprehensive study},
  author={Liang, Deron and Lu, Chia-Chi and Tsai, Chih-Fong and Shih, Guan-An},
  journal={European journal of operational research},
  volume={252},
  number={2},
  pages={561--572},
  year={2016},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="liang2016financial",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We map binary Bankrupt column to yes/no
- We rename the features to be shorter and have no special characters.
- We drop the constant column "Net_Income_Flag".
- We drop two duplicated features (same values as other features).
- Anomaly: the paper introducing the dataset has more features, but these do not seem to be in the dataset. 
- Anomaly: the data is temporal but all features are time-invariant. This might still have a latent, non-verifiable temporal component, which we ignore for our predictive IID task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Bankrupt",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Bankrupt",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/data.csv")

df.columns = [
    "Bankrupt",
    "ROA_C_Before_Interest_Depreciation",
    "ROA_A_Before_Interest_After_Tax",
    "ROA_B_Before_Interest_Depreciation_After_Tax",
    "Operating_Gross_Margin",
    "Realized_Sales_Gross_Margin",
    "Operating_Profit_Rate",
    "PreTax_Net_Interest_Rate",
    "AfterTax_Net_Interest_Rate",
    "NonIndustry_Income_Expenditure_Revenue",
    "Continuous_Interest_Rate_After_Tax",
    "Operating_Expense_Rate",
    "R&D_Expense_Rate",
    "Cash_Flow_Rate",
    "InterestBearing_Debt_Interest_Rate",
    "Tax_Rate_A",
    "Net_Value_Per_Share_B",
    "Net_Value_Per_Share_A",
    "Net_Value_Per_Share_C",
    "Persistent_EPS_Last_4_Seasons",
    "Cash_Flow_Per_Share",
    "Revenue_Per_Share",
    "Operating_Profit_Per_Share",
    "Net_Profit_Before_Tax_Per_Share",
    "Realized_Sales_Gross_Profit_Growth_Rate",
    "Operating_Profit_Growth_Rate",
    "AfterTax_Net_Profit_Growth_Rate",
    "Regular_Net_Profit_Growth_Rate",
    "Continuous_Net_Profit_Growth_Rate",
    "Total_Asset_Growth_Rate",
    "Net_Value_Growth_Rate",
    "Total_Asset_Return_Growth_Rate",
    "Cash_Reinvestment_Percent",
    "Current_Ratio",
    "Quick_Ratio",
    "Interest_Expense_Ratio",
    "Total_Debt_to_Net_Worth",
    "Debt_Ratio_Percent",
    "Net_Worth_to_Assets",
    "LongTerm_Fund_Suitability_Ratio_A",
    "Borrowing_Dependency",
    "Contingent_Liabilities_to_Net_Worth",
    "Operating_Profit_to_PaidIn_Capital",
    "Net_Profit_Before_Tax_to_PaidIn_Capital",
    "Inventory_Accounts_Receivable_to_Net_Value",
    "Total_Asset_Turnover",
    "Accounts_Receivable_Turnover",
    "Average_Collection_Days",
    "Inventory_Turnover_Rate",
    "Fixed_Assets_Turnover_Frequency",
    "Net_Worth_Turnover_Rate",
    "Revenue_Per_Person",
    "Operating_Profit_Per_Person",
    "Allocation_Rate_Per_Person",
    "Working_Capital_to_Total_Assets",
    "Quick_Assets_to_Total_Assets",
    "Current_Assets_to_Total_Assets",
    "Cash_to_Total_Assets",
    "Quick_Assets_to_Current_Liability",
    "Cash_to_Current_Liability",
    "Current_Liability_to_Assets",
    "Operating_Funds_to_Liability",
    "Inventory_to_Working_Capital",
    "Inventory_to_Current_Liability",
    "Current_Liabilities_to_Liability",
    "Working_Capital_to_Equity",
    "Current_Liabilities_to_Equity",
    "LongTerm_Liability_to_Current_Assets",
    "Retained_Earnings_to_Total_Assets",
    "Total_Income_to_Total_Expense",
    "Total_Expense_to_Assets",
    "Current_Asset_Turnover_Rate",
    "Quick_Asset_Turnover_Rate",
    "Working_Capital_Turnover_Rate",
    "Cash_Turnover_Rate",
    "Cash_Flow_to_Sales",
    "Fixed_Assets_to_Assets",
    "Current_Liability_to_Liability",
    "Current_Liability_to_Equity",
    "Equity_to_LongTerm_Liability",
    "Cash_Flow_to_Total_Assets",
    "Cash_Flow_to_Liability",
    "CFO_to_Assets",
    "Cash_Flow_to_Equity",
    "Current_Liability_to_Current_Assets",
    "Liability_Assets_Flag",
    "Net_Income_to_Total_Assets",
    "Total_Assets_to_GNP_Price",
    "NoCredit_Interval",
    "Gross_Profit_to_Sales",
    "Net_Income_to_Stockholders_Equity",
    "Liability_to_Equity",
    "DFL",
    "Interest_Coverage_Ratio",
    "Net_Income_Flag",
    "Equity_to_Liability",
]

cat_features = [
    "Bankrupt",
]
df["Bankrupt"] = df["Bankrupt"].map({0: "No", 1: "Yes"})
df[cat_features] = df[cat_features].astype("category")

df = df.drop(columns=[
    "Net_Income_Flag",
    "Current_Liability_to_Liability",
    "Current_Liability_to_Equity",
])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()